In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/severity/complaints_nlp.csv


In [2]:
import pandas as pd

df=pd.read_csv("/kaggle/input/severity/complaints_nlp.csv")
df.head()

/tmp/ipykernel_20/4146263574.py:3: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("/kaggle/input/severity/complaints_nlp.csv")


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,03/23/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,The Summer of XX/XX/2018 I was denied a mortga...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,NaN,NaN,Consent provided,Web,03/23/2019,Closed with explanation,Yes,NaN,3189109
1,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",VA,220XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3187982
2,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,770XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3187954
3,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,787XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3188091
4,03/22/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account information incorrect,There are many mistakes appear in my report wi...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,951XX,NaN,Consent provided,Web,03/22/2019,Closed with explanation,Yes,NaN,3188119


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

# Use the text + new severity labels
df_train = df.dropna(subset=["Consumer complaint narrative", "severity"]).copy()

texts = df_train["Consumer complaint narrative"].tolist()
labels = df_train["severity"].tolist()

label2id = {"low": 0, "medium": 1, "high": 2}
id2label = {0: "low", 1: "medium", 2: "high"}

df_train["label_id"] = df_train["severity"].map(label2id)

# Split dataset
train_df, val_df = train_test_split(
    df_train,
    test_size=0.1,
    stratify=df_train["label_id"],
    random_state=42
)

# Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["Consumer complaint narrative"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = Dataset.from_pandas(train_df[["Consumer complaint narrative", "label_id"]])
val_dataset   = Dataset.from_pandas(val_df[["Consumer complaint narrative", "label_id"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset = val_dataset.rename_column("label_id", "labels")

train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# Force GPU
if torch.cuda.is_available():
    model = model.to("cuda")
    print("🔥 Using GPU:", torch.cuda.get_device_name(0))

# Training settings
args = TrainingArguments(
    output_dir="/kaggle/working/severity_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("🚀 Training severity model...")
trainer.train()

# Save model
model.save_pretrained("/kaggle/working/severity_model")
tokenizer.save_pretrained("/kaggle/working/severity_model")
print("model saved")

2025-12-06 18:08:49.359421: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765044529.761295      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765044529.874436      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

KeyError: ['severity']

In [ ]:
import pandas as pd

def assign_severity(text):
    t = text.lower()

    # HIGH severity (money loss, fraud)
    high_keywords = [
        "fraud", "unauthorized", "scam", "identity theft",
        "stolen", "lost money", "charged twice", "deducted",
        "wrong amount", "not available", "legal action",
        "chargeback", "freeze", "security breach"
    ]
    if any(k in t for k in high_keywords):
        return "high"

    # MEDIUM severity (loan, payment, billing)
    medium_keywords = [
        "loan", "mortgage", "credit report", 
        "billing", "payment", "interest rate", 
        "fees", "escrow", "debt", "collection"
    ]
    if any(k in t for k in medium_keywords):
        return "medium"

    # LOW severity (general issues)
    return "low"


df["severity"] = df["Consumer complaint narrative"].fillna("").apply(assign_severity)
df = df[df["severity"].notnull()]
df.head()


In [ ]:
!nvidia-smi


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Model + Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Dataset
train_dataset = Dataset.from_pandas(train_df[["Consumer complaint narrative", "label_id"]])
val_dataset   = Dataset.from_pandas(val_df[["Consumer complaint narrative", "label_id"]])

def tokenize(batch):
    return tokenizer(batch["Consumer complaint narrative"], truncation=True, padding="max_length", max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
val_dataset   = val_dataset.rename_column("label_id", "labels")

train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
)
print("🔥 Model loaded")

# Training Arguments
args = TrainingArguments(
    output_dir="/kaggle/working/severity_model",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,                   # forces GPU half precision
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("🚀 Starting training (GPU REQUIRED!)")
trainer.train()

model.save_pretrained("/kaggle/working/severity_model")
tokenizer.save_pretrained("/kaggle/working/severity_model")
print("model saved")

In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="/kaggle/working/severity_model_clean",
    tokenizer="/kaggle/working/severity_model_clean"
)

tests = [
    "Someone stole my identity and took a loan in my name.",
    "My credit card was charged twice for the same item.",
    "How do I update my phone number?",
    "My loan payment is showing late even though I paid on time."
]

for t in tests:
    print(t, "→", clf(t)[0])


In [ ]:
from transformers import AutoConfig
config = AutoConfig.from_pretrained("/kaggle/working/severity_model")
print(config.id2label)


In [ ]:
label_mapping = {
    "LABEL_0": "Low Severity",
    "LABEL_1": "Medium Severity",
    "LABEL_2": "High Severity"
}

from transformers import pipeline

classifier = pipeline("text-classification", model="/kaggle/working/severity_model", device="cpu")

tests = [
    "Someone stole my identity and took a loan in my name.",
    "My credit card was charged twice for the same item.",
    "How do I update my phone number?",
    "My loan payment is showing late even though I paid on time."
]

for t in tests:
    pred = classifier(t)[0]
    severity = label_mapping[pred["label"]]
    print({"text": t, "severity": severity, "score": pred["score"]})


In [ ]:
model.config.id2label = {
    0: "Low Severity",
    1: "Medium Severity",
    2: "High Severity"
}

model.config.label2id = {
    "Low Severity": 0,
    "Medium Severity": 1,
    "High Severity": 2
}

model.save_pretrained("/kaggle/working/severity_model_clean")
tokenizer.save_pretrained("/kaggle/working/severity_model_clean")
print("model saved")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/severity_model_clean_k", 'zip', "/kaggle/working/severity_model_clean")


In [ ]:
from IPython.display import FileLink
FileLink("/kaggle/working/severity_model_clean_k.zip")
